In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
from pyspark.sql.functions import (
    col, sum as spark_sum, count, window
)
import pandas as pd
import os
import shutil
import time
from datetime import datetime

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [3]:
# Mendefinisikan skema data pesanan
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membuat folder streaming
folder_stream = "tugas10_stream"
shutil.rmtree(folder_stream, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

# Membaca data secara streaming
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

print("Folder streaming berhasil dibuat!")
print("Status streaming:", df_stream.isStreaming)

Folder streaming berhasil dibuat!
Status streaming: True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [4]:
# Menambahkan kolom total_harga
df_stream_harga = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan"))

# Mengelompokkan data berdasarkan nama cabang
agregasi_cabang = df_stream_harga.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan"))

# Menjalankan streaming query
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi cabang berhasil dijalankan!")

Query agregasi cabang berhasil dijalankan!


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [5]:
# Membaca ulang data streaming dari folder yang sama
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_stream)

# Menghitung total harga
df_waktu_harga = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan"))

# Agregasi berdasarkan window waktu 2 menit
agregasi_waktu = df_waktu_harga.groupBy(
    window(col("waktu_pesan"), "2 minutes")).agg(
    spark_sum("total_harga").alias("total_pendapatan"))

# Menjalankan streaming query kedua
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi window waktu berhasil dijalankan!")

Query agregasi window waktu berhasil dijalankan!


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [7]:
# Bach 1
waktu1 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Matcha Latte"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 15000, 22000],
    "waktu_pesan": [waktu1] * 3
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

time.sleep(6)
print("Hasil agregasi Batch 1:")
spark.sql("SELECT * FROM agregasi_cabang").show()

Hasil agregasi Batch 1:
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           15000|             1|
|   Magelang|          102000|             2|
+-----------+----------------+--------------+



In [9]:
# Bach 2
waktu2 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Solo"],
    "nama_menu": ["Cappuccino", "Kopi Tubruk", "Chocolate"],
    "qty": [1, 2, 2],
    "harga_satuan": [25000, 16000, 20000],
    "waktu_pesan": [waktu2] * 3
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

time.sleep(6)
print("Hasil agregasi Batch 2:")
spark.sql("SELECT * FROM agregasi_cabang").show()

Hasil agregasi Batch 2:
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           55000|             2|
| Yogyakarta|           32000|             1|
|   Magelang|          127000|             3|
+-----------+----------------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [11]:
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query Parquet berhasil dijalankan!")

Streaming query Parquet berhasil dijalankan!


In [12]:
waktu4 = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Yogyakarta"],
    "nama_menu": ["Vanilla Latte", "Mocha"],
    "qty": [2, 1],
    "harga_satuan": [24000, 26000],
    "waktu_pesan": [waktu4, waktu4]
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

time.sleep(8)

print("Batch baru berhasil dikirim!")

Batch baru berhasil dikirim!


In [13]:
!ls tugas10_output

part-00000-1658ba40-8c63-4c57-b17e-956f259a5322-c000.snappy.parquet
part-00000-8d0d3d32-fd1a-4ce6-9e9b-bd86fba5167c-c000.snappy.parquet
part-00001-b759f993-7130-49b0-9cc9-5c81d060f3e1-c000.snappy.parquet
_spark_metadata


In [15]:
df_parquet = spark.read.parquet("tugas10_output")

df_parquet.show(truncate=False)

+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu    |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------+---+------------+-------------------+
|P004      |Magelang   |Cappuccino   |1  |25000       |2026-11-05 08:47:05|
|P005      |Yogyakarta |Kopi Tubruk  |2  |16000       |2026-11-05 08:47:05|
|P006      |Solo       |Chocolate    |2  |20000       |2026-11-05 08:47:05|
|P001      |Magelang   |Kopi Susu    |2  |18000       |2026-11-05 08:45:59|
|P002      |Solo       |Americano    |1  |15000       |2026-11-05 08:45:59|
|P003      |Magelang   |Matcha Latte |3  |22000       |2026-11-05 08:45:59|
|P010      |Magelang   |Vanilla Latte|2  |24000       |2026-11-05 08:51:31|
|P011      |Yogyakarta |Mocha        |1  |26000       |2026-11-05 08:51:31|
+----------+-----------+-------------+---+------------+-------------------+



In [16]:
query1.stop()
query2.stop()
query3.stop()

spark.stop()
print("Seluruh streaming query berhasil dihentikan")

Seluruh streaming query berhasil dihentikan


**Eksplorasi**

In [17]:
spark = SparkSession.builder \
    .appName("EksplorasiKopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuka!")

SparkSession berhasil dibuka!


In [19]:
from pyspark.sql.functions import col, sum as spark_sum, count
df_parquet = spark.read.parquet("tugas10_output")
df_eksplorasi = df_parquet.withColumn(
    "total_harga", col("qty") * col("harga_satuan"))

# Query 1: Pendapatan dan jumlah pesanan tiap cabang
hasil_cabang = df_eksplorasi.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan"))

hasil_cabang.orderBy(col("total_pendapatan").desc()).show()

# Query 2: Menu paling banyak terjual
hasil_menu = df_eksplorasi.groupBy("nama_menu").agg(
    spark_sum("qty").alias("total_terjual"))

hasil_menu.orderBy(col("total_terjual").desc()).show()

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          175000|             4|
| Yogyakarta|           58000|             2|
|       Solo|           55000|             2|
+-----------+----------------+--------------+

+-------------+-------------+
|    nama_menu|total_terjual|
+-------------+-------------+
| Matcha Latte|            3|
|    Chocolate|            2|
|  Kopi Tubruk|            2|
|    Kopi Susu|            2|
|Vanilla Latte|            2|
|   Cappuccino|            1|
|    Americano|            1|
|        Mocha|            1|
+-------------+-------------+



In [20]:
spark.stop()
print("SparkSession berhasil ditutup!")

SparkSession berhasil ditutup!
